In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import statistics as st
from resourcecode import Client
from geopy import distance
import matplotlib.pyplot as plt

client = Client()

In [18]:
df = pd.read_csv("/Users/leitejard/prot3d/data/coord_time.csv", index_col=0)
print(df)

          point                 date   latitude  longitude  \
1     Rozegat 1  2019-02-21T00:00:00  48.320383  -4.387650   
2     Rozegat 1  2022-04-15T00:00:00  48.320383  -4.387650   
3     Rozegat 1  2023-03-18T00:00:00  48.320383  -4.387650   
4     Rozegat 2  2019-02-21T00:00:00  48.321950  -4.371167   
5     Rozegat 2  2022-04-15T00:00:00  48.321950  -4.371167   
..          ...                  ...        ...        ...   
86  Trevignon 3  2022-03-25T00:00:00  47.791583  -3.885767   
87  Trevignon 3  2023-03-16T00:00:00  47.791583  -3.885767   
88     Meaban 3  2019-02-20T00:00:00  47.525800  -2.934483   
89     Meaban 3  2022-03-24T00:00:00  47.525800  -2.934483   
90     Meaban 3  2023-03-15T00:00:00  47.525800  -2.934483   

                  date0  
1   2018-11-23T00:00:00  
2   2022-01-15T00:00:00  
3   2022-12-18T00:00:00  
4   2018-11-23T00:00:00  
5   2022-01-15T00:00:00  
..                  ...  
86  2021-12-25T00:00:00  
87  2022-12-16T00:00:00  
88  2018-11-22T00:00:

In [7]:
#Let's use the unstructured grid model in which the resolution can reach 200m
url="https://tds1.ifremer.fr/thredds/dodsC/MARC-MANCHEGASCOGNENORD-WW3_NORGAS_UG-FOR_FULL_TIME_SERIE"
remote_data = xr.open_dataset(url)
#let's get the latitude and longitudes, bc sometimes the vectors do not have the same lengths, we will force them into a df and fill in na's 
lat=pd.DataFrame(remote_data.latitude) 
long=pd.DataFrame(remote_data.longitude)

distdf=pd.concat([lat, long], ignore_index=True, axis=1).fillna(0)
distdf.columns = ['lat', 'long']
distdf['distance'] = np.nan #create a distance variable to calculate de distance from the sampling point to the closes grid point in the model
print(distdf)

              lat      long  distance
0       51.632282 -8.535430       NaN
1       51.654434 -8.500304       NaN
2       51.659142 -8.534190       NaN
3       51.619572 -8.484489       NaN
4       51.605419 -8.536670       NaN
...           ...       ...       ...
110799  48.437500 -7.500000       NaN
110800  50.250000 -7.500000       NaN
110801  50.750000 -7.500000       NaN
110802  50.812500 -7.500000       NaN
110803  50.875000 -7.500000       NaN

[110804 rows x 3 columns]


In [19]:
df['closest_point'] = None
df['distance_to_point'] = None
#calculate the closest point of the grid to every sampling point
for index, i in df.iterrows():
        mylat = i['latitude']
        mylong = i['longitude']
        print(i)
        # we take all model's coordinates and calculate de distance from each grid point to the sampling point
        dist = distdf.apply(lambda row: distance.distance((mylat,mylong),(row.lat, row.long)).m, axis=1)
        # on recupere l'indice du noeud de la distance minimale
        id_node=dist.idxmin()
        distance_to_point = dist.min()
        df.at[index, 'closest_point'] = id_node
        df.at[index, 'distance_to_point'] = distance_to_point
print(df)

point                          Rozegat 1
date                 2019-02-21T00:00:00
latitude                       48.320383
longitude                       -4.38765
date0                2018-11-23T00:00:00
closest_point                       None
distance_to_point                   None
Name: 1, dtype: object
point                          Rozegat 1
date                 2022-04-15T00:00:00
latitude                       48.320383
longitude                       -4.38765
date0                2022-01-15T00:00:00
closest_point                       None
distance_to_point                   None
Name: 2, dtype: object
point                          Rozegat 1
date                 2023-03-18T00:00:00
latitude                       48.320383
longitude                       -4.38765
date0                2022-12-18T00:00:00
closest_point                       None
distance_to_point                   None
Name: 3, dtype: object
point                          Rozegat 2
date                 2019-02-

In [15]:
times=pd.to_datetime(remote_data.time.data)

In [ ]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydata=remote_data.hs[id_tdeb:id_tfin,row['closest_point']]
        dft=pd.DataFrame(mydata,index=mydata.time.data)
        hs_mean = dft.mean()
        hs_median = dft.median()
        hs_sd = np.std(dft[0])
        hs_min = dft.min()
        hs_max = dft.max()
        fulldf.append([row['point'],row['date'],float(hs_mean), float(hs_median), hs_sd, float(hs_min), float(hs_max)])

columns = ["point", "date", "hs_mean", "hs_median","hs_sd", "hs_min", "hs_max"]
hsdf = pd.DataFrame(fulldf, columns=columns)
print(hsdf)

In [20]:
hsdf.to_csv("/Users/leitejard/prot3d/data/hs.csv")

In [ ]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydata=remote_data.fp[id_tdeb:id_tfin,row['closest_point']]
        dft=pd.DataFrame(mydata,index=mydata.time.data)
        fp_mean = dft.mean()
        fp_median = dft.median()
        fp_sd = np.std(dft)
        fp_min = dft.min()
        fp_max = dft.max()
        fulldf.append([row['point'],row['date'],float(fp_mean[0]), float(fp_median[0]), float(fp_sd[0]), float(fp_min[0]), float(fp_max[0])])

columns = ["point", "date", "fp_mean", "fp_median","fp_sd", "fp_min", "fp_max"]
fpdf = pd.DataFrame(fulldf, columns=columns)
print(fpdf)

In [32]:
fpdf.to_csv("/Users/leitejard/prot3d/data/fp.csv")

In [ ]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydata=remote_data.ucur[id_tdeb:id_tfin,row['closest_point']]
        dft=pd.DataFrame(mydata,index=mydata.time.data)
        ucur_mean = dft.mean()
        ucur_median = dft.median()
        ucur_sd = np.std(dft)
        ucur_min = dft.min()
        ucur_max = dft.max()
        fulldf.append([row['point'],row['date'],float(ucur_mean[0]), float(ucur_median[0]), float(ucur_sd[0]), float(ucur_min[0]), float(ucur_max[0])])

columns = ["point", "date", "ucur_mean", "ucur_median","ucur_sd", "ucur_min", "ucur_max"]
ucurdf = pd.DataFrame(fulldf, columns=columns)
print(ucurdf)

In [25]:
ucurdf.to_csv("/Users/leitejard/prot3d/data/ucurd.csv")

In [20]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydatav=remote_data.vcur[id_tdeb:id_tfin,row['closest_point']]
        dftv=pd.DataFrame(mydatav,index=mydatav.time.data)
        mydatau=remote_data.ucur[id_tdeb:id_tfin,row['closest_point']]
        dftu=pd.DataFrame(mydatau,index=mydatau.time.data)
        dft = np.sqrt((dftv**2) + (dftu**2))
        cur_mean = dft.mean()
        cur_median = dft.median()
        cur_sd = np.std(dft)
        cur_min = dft.min()
        cur_max = dft.max()
        fulldf.append([row['point'],row['date'],float(cur_mean), float(cur_median), float(cur_sd), float(cur_min), float(cur_max)])

columns = ["point", "date", "cur_mean", "cur_median","cur_sd", "cur_min", "cur_max"]
curdf = pd.DataFrame(fulldf, columns=columns)
print(curdf)

Rozegat 1


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1540211800.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(cur_mean), float(cur_median), float(cur_sd), float(cur_min), float(cur_max)])


Rozegat 1


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1540211800.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(cur_mean), float(cur_median), float(cur_sd), float(cur_min), float(cur_max)])


Rozegat 1


...

/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1540211800.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(cur_mean), float(cur_median), float(cur_sd), float(cur_min), float(cur_max)])


Meaban 3


Error:curl error: Failure when receiving data from the peer


          point                 date  cur_mean  cur_median    cur_sd  \
0     Rozegat 1  2019-02-21T00:00:00  0.154620    0.160312  0.073163   
1     Rozegat 1  2022-04-15T00:00:00  0.140040    0.136015  0.077545   
2     Rozegat 1  2023-03-18T00:00:00  0.141248    0.140357  0.076421   
3     Rozegat 2  2019-02-21T00:00:00  0.222554    0.220227  0.110175   
4     Rozegat 2  2022-04-15T00:00:00  0.195268    0.191050  0.111743   
..          ...                  ...       ...         ...       ...   
85  Trevignon 3  2022-03-25T00:00:00  0.051699    0.050990  0.016325   
86  Trevignon 3  2023-03-16T00:00:00  0.051123    0.050000  0.015194   
87     Meaban 3  2019-02-20T00:00:00  0.119012    0.100499  0.073008   
88     Meaban 3  2022-03-24T00:00:00  0.104927    0.100499  0.052137   
89     Meaban 3  2023-03-15T00:00:00  0.103501    0.100499  0.051074   

     cur_min   cur_max  
0   0.000000  0.345398  
1   0.000000  0.317805  
2   0.010000  0.320156  
3   0.000000  0.548179  
4   0.0100

/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1540211800.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(cur_mean), float(cur_median), float(cur_sd), float(cur_min), float(cur_max)])


In [22]:
curdf.to_csv("/Users/leitejard/prot3d/data/cur.csv")

In [25]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydatav=remote_data.vwnd[id_tdeb:id_tfin,row['closest_point']]
        dftv=pd.DataFrame(mydatav,index=mydatav.time.data)
        mydatau=remote_data.uwnd[id_tdeb:id_tfin,row['closest_point']]
        dftu=pd.DataFrame(mydatau,index=mydatau.time.data)
        dft = np.sqrt((dftv**2) + (dftu**2))
        wnd_mean = dft.mean()
        wnd_median = dft.median()
        wnd_sd = np.std(dft)
        wnd_min = dft.min()
        wnd_max = dft.max()
        fulldf.append([row['point'],row['date'],float(wnd_mean), float(wnd_median), float(wnd_sd), float(wnd_min), float(wnd_max)])

columns = ["point", "date", "wnd_mean", "wnd_median","wnd_sd", "wnd_min", "wnd_max"]
wnddf = pd.DataFrame(fulldf, columns=columns)
print(wnddf)

Rozegat 1


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1800605138.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(wnd_mean), float(wnd_median), float(wnd_sd), float(wnd_min), float(wnd_max)])


Rozegat 1


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1800605138.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(wnd_mean), float(wnd_median), float(wnd_sd), float(wnd_min), float(wnd_max)])


Rozegat 1


...

Error:curl error: Failure when receiving data from the peer
/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1800605138.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(wnd_mean), float(wnd_median), float(wnd_sd), float(wnd_min), float(wnd_max)])


Meaban 3


Error:curl error: Failure when receiving data from the peer


          point                 date  wnd_mean  wnd_median    wnd_sd  \
0     Rozegat 1  2019-02-21T00:00:00  5.037703    4.634620  2.855769   
1     Rozegat 1  2022-04-15T00:00:00  4.711501    4.204760  2.529203   
2     Rozegat 1  2023-03-18T00:00:00  5.578256    4.970915  3.093202   
3     Rozegat 2  2019-02-21T00:00:00  4.947509    4.554119  2.801372   
4     Rozegat 2  2022-04-15T00:00:00  4.630644    4.123106  2.486256   
..          ...                  ...       ...         ...       ...   
85  Trevignon 3  2022-03-25T00:00:00  5.864918    5.435531  2.972939   
86  Trevignon 3  2023-03-16T00:00:00  6.585116    5.957767  3.435741   
87     Meaban 3  2019-02-20T00:00:00  6.678247    6.005414  3.533050   
88     Meaban 3  2022-03-24T00:00:00  6.245995    5.752390  3.118438   
89     Meaban 3  2023-03-15T00:00:00  7.167506    6.580274  3.545452   

     wnd_min    wnd_max  
0   0.200000  14.911070  
1   0.300000  13.753909  
2   0.100000  13.790576  
3   0.200000  14.697278  
4   0

/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/1800605138.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(wnd_mean), float(wnd_median), float(wnd_sd), float(wnd_min), float(wnd_max)])


In [26]:
wnddf.to_csv("/Users/leitejard/prot3d/data/wnd.csv")

In [23]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydatav=remote_data.vabr[id_tdeb:id_tfin,row['closest_point']]
        dftv=pd.DataFrame(mydatav,index=mydatav.time.data)
        mydatau=remote_data.uabr[id_tdeb:id_tfin,row['closest_point']]
        dftu=pd.DataFrame(mydatau,index=mydatau.time.data)
        dft = np.sqrt((dftv**2) + (dftu**2))
        abr_mean = dft.mean()
        abr_median = dft.median()
        abr_sd = np.std(dft)
        abr_min = dft.min()
        abr_max = dft.max()
        fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])

columns = ["point", "date", "abr_mean", "abr_median","abr_sd", "abr_min", "abr_max"]
abrdf = pd.DataFrame(fulldf, columns=columns)
print(abrdf)

Rozegat 1


Error:curl error: Failure when receiving data from the peer
/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/2963959893.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])


Rozegat 1


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/2963959893.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])


Rozegat 1


...

/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/2963959893.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])


Meaban 3


/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/2963959893.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])


Meaban 3
          point                 date  abr_mean  abr_median    abr_sd  \
0     Rozegat 1  2019-02-21T00:00:00  0.027487    0.022361  0.016487   
1     Rozegat 1  2022-04-15T00:00:00  0.024867    0.014142  0.016159   
2     Rozegat 1  2023-03-18T00:00:00  0.030143    0.028284  0.018410   
3     Rozegat 2  2019-02-21T00:00:00  0.021768    0.014142  0.013960   
4     Rozegat 2  2022-04-15T00:00:00  0.019497    0.014142  0.014133   
..          ...                  ...       ...         ...       ...   
85  Trevignon 3  2022-03-25T00:00:00  0.376949    0.347131  0.224561   
86  Trevignon 3  2023-03-16T00:00:00  0.362538    0.304795  0.216858   
87     Meaban 3  2019-02-20T00:00:00  0.267037    0.233238  0.142443   
88     Meaban 3  2022-03-24T00:00:00  0.288917    0.277849  0.149495   
89     Meaban 3  2023-03-15T00:00:00  0.284348    0.264008  0.142407   

     abr_min   abr_max  
0   0.000000  0.114018  
1   0.000000  0.164012  
2   0.000000  0.144222  
3   0.000000  0.098489  
4

/Users/leitejard/micromamba/envs/env_pywave/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_49735/2963959893.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(abr_mean), float(abr_median), float(abr_sd), float(abr_min), float(abr_max)])


In [24]:
abrdf.to_csv("/Users/leitejard/prot3d/data/abr.csv")

In [7]:
fulldf = []
for index, row in df.iterrows():
        # Extract values from the DataFrame for the current row
        print(row['point'])
        id_tdeb=times.searchsorted(np.datetime64(row['date0']))
        id_tfin = times.searchsorted(np.datetime64(row['date']))
        mydata=remote_data.dpt[id_tdeb:id_tfin,row['closest_point']]
        dft=pd.DataFrame(mydata,index=mydata.time.data)
        dpt_mean = dft.mean()
        dpt_median = dft.median()
        dpt_sd = np.std(dft[0])
        dpt_min = dft.min()
        dpt_max = dft.max()
        fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])

columns = ["point", "date", "dpt_mean", "dpt_median","dpt_sd", "dpt_min", "dpt_max"]
dptdf = pd.DataFrame(fulldf, columns=columns)
print(dptdf)

Rozegat 1


Error:curl error: Failure when receiving data from the peer
/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_82851/3373394322.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])


Rozegat 1


/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_82851/3373394322.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])


Rozegat 1


...

/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_82851/3373394322.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])


Meaban 3


/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_82851/3373394322.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])


Meaban 3
          point                 date   dpt_mean  dpt_median    dpt_sd  \
0     Rozegat 1  2019-02-21T00:00:00   4.959028         5.0  1.651616   
1     Rozegat 1  2022-04-15T00:00:00   4.976389         5.0  1.577298   
2     Rozegat 1  2023-03-18T00:00:00   4.976852         5.0  1.580677   
3     Rozegat 2  2019-02-21T00:00:00   4.771528         5.0  1.656706   
4     Rozegat 2  2022-04-15T00:00:00   4.782407         5.0  1.579781   
..          ...                  ...        ...         ...       ...   
85  Trevignon 3  2022-03-25T00:00:00  15.643982        15.5  1.196101   
86  Trevignon 3  2023-03-16T00:00:00  15.643518        15.5  1.177137   
87     Meaban 3  2019-02-20T00:00:00   9.330556         9.5  1.332462   
88     Meaban 3  2022-03-24T00:00:00   9.353472         9.5  1.293301   
89     Meaban 3  2023-03-15T00:00:00   9.353241         9.5  1.273206   

    dpt_min  dpt_max  
0       1.5      8.5  
1       2.0      8.0  
2       1.5      8.0  
3       1.0      8.0  

/var/folders/1x/txzh8wxd4v1cb8dsb4lkkwqm0000gp/T/ipykernel_82851/3373394322.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  fulldf.append([row['point'],row['date'],float(dpt_mean), float(dpt_median), dpt_sd, float(dpt_min), float(dpt_max)])


In [8]:
dptdf.to_csv("/Users/leitejard/prot3d/data/dpt.csv")